# V117 - Adversarial validation

Pre-registered plan: every experiment assumes train and test users come from one distribution, but nobody tested it. A classifier separating train from test users on all features measures covariate shift directly. AUC near 0.5 closes the question. AUC clearly above 0.5 opens importance weighting as the next step, so feature importances are recorded to name the shifted variables. Labels never enter.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import StratifiedKFold

SEED = 4646
N_SPLITS = 5
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v117")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Can features tell train from test

Balanced labels are split membership. AUC with a bootstrap interval decides. Importances name the shift if any.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv", parse_dates=["timestamp"])
assess = pd.read_csv(DATA / "user_assessments.csv")
n_train, n_test = len(train), len(test_ids)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
NCH = np.array([len(by_user.get(u, [])) for u in all_users])[:, None]
chats["timestamp"] = pd.to_datetime(chats["timestamp"], errors="coerce")
first_ts = chats.sort_values("timestamp").groupby("user_id")["timestamp"].first()
TS = pd.to_datetime(pd.Series([first_ts.get(u, pd.NaT) for u in all_users]))
TORD = (TS - TS.min()).dt.total_seconds().fillna(-1).to_numpy()[:, None]
X = np.hstack([W, A, NCH / (NCH.mean() + 1e-9), TORD / (np.abs(TORD).max() + 1e-9)])
print("features:", X.shape)
y = np.array([0] * n_train + [1] * n_test)

from sklearn.metrics import roc_auc_score
oof = np.zeros(len(y))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for tr, va in skf.split(X, y):
    c = LogisticRegression(max_iter=1000)
    c.fit(X[tr], y[tr])
    oof[va] = c.predict_proba(X[va])[:, 1]
auc = roc_auc_score(y, oof)
lo, hi = np.quantile([roc_auc_score(y[b], oof[b]) for b in [rng.choice(len(y), len(y), replace=True) for _ in range(1000)]], [0.025, 0.975])
print(f"adversarial AUC={auc:.4f} CI95=[{lo:.4f},{hi:.4f}]")
print("SHIFT PRESENT: open importance weighting" if lo > 0.55 else "NO SHIFT: train and test are one distribution for these features")